In [6]:
from ultralytics import YOLO
def process_banana_224(img, img_idx=None, debug=False):
    h_orig, w_orig = img.shape[:2]

    results = model_yolo.predict(source=img, verbose=False
                                 ,classes=[46])
    # )
    result = results[0]

    if result.masks is None or len(result.masks.xy) == 0:
        print(f"(!) Ảnh {img_idx}: YOLO không detect")
        return None, None

    # TÌM MASK LỚN NHẤT (dựa trên diện tích bounding box)
    boxes_np = result.boxes.xyxy.cpu().numpy()
    areas = [(box[2] - box[0]) * (box[3] - box[1]) for box in boxes_np]
    idx = np.argmax(areas)

    # MASK YOLO: Dùng tọa độ Polygon
    polygon = result.masks.xy[idx].astype(np.int32)
    mask = np.zeros((h_orig, w_orig), dtype=np.uint8)
    cv2.fillPoly(mask, [polygon], 255)

    x1, y1, x2, y2 = result.boxes.xyxy[idx].cpu().numpy().astype(int)

    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w_orig, x2), min(h_orig, y2)

    banana_only = cv2.bitwise_and(img, img, mask=mask)
    cropped = banana_only[y1:y2, x1:x2]

    if cropped.size == 0:
        print(f"(!) Ảnh {img_idx}: crop lỗi (kích thước = 0)")
        return None, None

    ch, cw = cropped.shape[:2]
    max_side = max(ch, cw)

    # Khởi tạo ảnh vuông nền đen (PADDING)
    square_img = np.zeros((max_side, max_side, 3), dtype=np.uint8)
    y_off, x_off = (max_side - ch) // 2, (max_side - cw) // 2
    square_img[y_off:y_off+ch, x_off:x_off+cw] = cropped

    final_img = cv2.resize(square_img, (224, 224), interpolation=cv2.INTER_AREA)

    # Crop và pad cho mask tương tự như ảnh
    mask_crop = mask[y1:y2, x1:x2]
    square_mask = np.zeros((max_side, max_side), dtype=np.uint8)
    square_mask[y_off:y_off+ch, x_off:x_off+cw] = mask_crop

    # Resize mask (KHÔNG dùng INTER_AREA, dùng NEAREST là chính xác)
    final_mask = cv2.resize(square_mask, (224, 224), interpolation=cv2.INTER_NEAREST)

    # ================= DEBUG =================
    if debug:
        fig, axs = plt.subplots(1, 5, figsize=(16,4))
        axs[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axs[0].set_title("Original")

        axs[1].imshow(mask, cmap='gray')
        axs[1].set_title("Mask")

        axs[2].imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
        axs[2].set_title("Cropped")

        axs[3].imshow(cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB))
        axs[3].set_title("Final 224")

        overlay = final_img.copy()
        overlay[final_mask == 0] = [0, 0, 255]

        axs[4].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axs[4].set_title("Test mask after resize")

        for ax in axs:
            ax.axis("off")

        plt.suptitle(f"YOLO | img_idx={img_idx}")
        plt.show()

        print("final image size:", final_img.shape)

    return final_img, final_mask

model_yolo = YOLO('yolov8x-seg.pt')

In [7]:
import gradio as gr
import cv2
import numpy as np
import joblib
import os
import importlib.util 
import pandas as pd # Thêm thư viện pandas để tạo bảng

BASE_DIR = os.getcwd()
MODELS_DIR = os.path.abspath(os.path.join(BASE_DIR, "../models"))

# Thêm biến toàn cục chứa hàm trích xuất
current_model = None
current_scaler = None
current_extract_fn = None  

def get_model_folders():
    if not os.path.exists(MODELS_DIR):
        os.makedirs(MODELS_DIR)
        return []
    return [d for d in os.listdir(MODELS_DIR) if os.path.isdir(os.path.join(MODELS_DIR, d))]

def load_selected_model(folder_name):
    global current_model, current_scaler, current_extract_fn
    
    if not folder_name:
        return "⚠️ Chưa chọn thư mục."
        
    folder_path = os.path.join(MODELS_DIR, folder_name)
    
    try:
        all_files = os.listdir(folder_path)
        
        # 1. TÌM VÀ NẠP MODEL + SCALER
        model_file = next((f for f in all_files if "model" in f.lower() and f.endswith(".pkl")), None)
        scaler_file = next((f for f in all_files if "scaler" in f.lower() and f.endswith(".pkl")), None)
        
        if not model_file or not scaler_file:
            return f"❌ Lỗi: Thiếu file model hoặc scaler trong thư mục {folder_name}."

        current_model = joblib.load(os.path.join(folder_path, model_file))
        current_scaler = joblib.load(os.path.join(folder_path, scaler_file))
        
        # 2. TÌM VÀ NẠP ĐỘNG FILE PYTHON (banana_features.py)
        py_file = next((f for f in all_files if f.endswith(".py")), None)
        if not py_file:
            return f"❌ Lỗi: Không tìm thấy file trích xuất đặc trưng (.py) trong thư mục {folder_name}."
            
        py_path = os.path.join(folder_path, py_file)
        
        # Khởi tạo module từ đường dẫn file
        spec = importlib.util.spec_from_file_location("dynamic_features", py_path)
        feature_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(feature_module)
        
        # Gắn hàm extract_combined_features (từ file vừa nạp) vào biến toàn cục
        if hasattr(feature_module, 'extract_combined_features'):
            current_extract_fn = feature_module.extract_combined_features
        elif hasattr(feature_module, 'compute_feature'):
            current_extract_fn = feature_module.compute_feature
        else:
            return f"❌ Lỗi: File {py_file} không chứa hàm 'extract_combined_features'."

        return f"✅ Đã tải thành công Pipeline từ: {folder_name}"
    except Exception as e:
        return f"❌ Lỗi: {str(e)}"

def predict_gradio(image):
    if image is None:
        return "<p style='text-align: center; color: gray;'>Chưa có ảnh</p>"
    
    if current_model is None or current_scaler is None or current_extract_fn is None:
        return "<p style='text-align: center; color: red;'>Lỗi: Vui lòng chọn Model</p>"
        
    reverse_label_map = {0: "Overripe", 1: "Ripe", 2: "Unripe"}
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    try:
        final_img, final_mask = process_banana_224(img_bgr)
        # trường hợp không phát hiện chuối 
        if final_img is None or final_mask is None:
             return "<div style='padding: 20px; text-align: center; background-color: #fee2e2; border-radius: 8px;'><p style='color: #ef4444; font-weight: bold; margin: 0;'>⚠️ Không phát hiện thấy quả chuối nào trong ảnh!</p><p style='color: #7f1d1d; font-size: 14px; margin-top: 5px;'>Vui lòng thử lại với một bức ảnh khác rõ ràng hơn.</p></div>"

        features = current_extract_fn(final_img, mask=final_mask)
        features_scaled = current_scaler.transform(np.array(features).reshape(1, -1))
        
        classes = current_model.classes_
        probabilities = current_model.predict_proba(features_scaled)[0]
        
        # Gộp và sắp xếp kết quả % từ cao xuống thấp
        results = []
        for cls, prob in zip(classes, probabilities):
            label_name = reverse_label_map.get(int(cls), f"Class {cls}")
            results.append((label_name, float(prob) * 100))
            
        results.sort(key=lambda x: x[1], reverse=True)

        # Vẽ HTML cho thanh tiến độ
        html_content = "<div style='padding: 10px; font-family: sans-serif;'>"
        for label, percent in results:
            html_content += f"""
            <div style='margin-bottom: 18px;'>
                <div style='display: flex; justify-content: space-between; margin-bottom: 6px; font-size: 16px; font-weight: 600; color: #333;'>
                    <span>{label}</span>
                    <span>{percent:.2f}%</span>
                </div>
                <div style='width: 100%; background-color: #e5e7eb; border-radius: 8px; height: 12px; overflow: hidden;'>
                    <div style='width: {percent}%; background-color: #f97316; height: 100%; transition: width 0.5s ease-in-out;'></div>
                </div>
            </div>
            """
        html_content += "</div>"
        
        return html_content
        
    except Exception as e:
        return f"<p style='color: red;'>Lỗi xử lý: {str(e)}</p>"

with gr.Blocks(title="Phân loại độ chín của chuối") as demo:
    gr.Markdown("<h1 style='text-align: center;'>🍌 HỆ THỐNG PHÂN LOẠI ĐỘ CHÍN CỦA CHUỐI</h1>")
    
    with gr.Row():
        with gr.Column(scale=2):
            image_input = gr.Image(label="🖼️ Khung tải ảnh", height=420)
            predict_btn = gr.Button("🔍 Phân tích ảnh", variant="primary")
            
        with gr.Column(scale=1):
            model_dropdown = gr.Dropdown(
                choices=get_model_folders(), 
                label="📁 Chọn Model (Thư mục)"
            )
            status_text = gr.Textbox(label="Trạng thái Model", interactive=False, value="Vui lòng chọn thư mục model.")
            
            label_output = gr.HTML(label="📊 Kết quả Phân tích")
    model_dropdown.change(
        fn=load_selected_model, 
        inputs=model_dropdown, 
        outputs=status_text
    )
    
    predict_btn.click(
        fn=predict_gradio, 
        inputs=image_input, 
        outputs=label_output
    )

if __name__ == "__main__":
    demo.launch(debug=True, share=True, theme=gr.themes.Monochrome())

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator KNeighborsClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle e

Keyboard interruption in main thread... closing server.
